# Construindo banco de dados das amostras de vigas

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import itertools
import numpy as np
import openpyxl
#!pip install parepy-toolbox
from parepy_toolbox import sampling_algorithm_structural_analysis
from obj import momento_limite_armadura_simples, area_aco_flexao_simples, momento_resistente_secao_sem_cor, obj_mestrado_victor

# Planejamento experimental

In [2]:
# Definindo os níveis das variáveis
h_levels = np.linspace(0.30, 0.70, 3)
b_levels = np.linspace(0.14, 0.35, 3)
f_levels = np.linspace(20000, 50000, 3)

# Gerando o planejamento fatorial completo com itertools.product
combinacoes = list(itertools.product(h_levels, b_levels, f_levels))

# Convertendo para DataFrame
df = pd.DataFrame(combinacoes, columns=['h', 'b_w', 'f_ck'])
df

,h,b_w,f_ck
0,0.3,0.140,20000.0
1,0.3,0.140,35000.0
2,0.3,0.140,50000.0
3,0.3,0.245,20000.0
4,0.3,0.245,35000.0
5,0.3,0.245,50000.0
6,0.3,0.350,20000.0
7,0.3,0.350,35000.0
8,0.3,0.350,50000.0
9,0.5,0.140,20000.0


# Criando o banco de armaduras

In [3]:
b_w = []
h = []
f_ck = []
m_rdlim = []
pho_s = []
a_s = []

for i in range(len(df)):
    b_w.append(df['b_w'][i])
    h.append(df['h'][i])
    f_ck.append(df['f_ck'][i])
    m_rdlim.append(momento_limite_armadura_simples(df['b_w'][i], df['h'][i], df['f_ck'][i]))
    a_saux, pho_saux = area_aco_flexao_simples(m_rdlim[i], df['b_w'][i], df['h'][i], df['f_ck'][i])
    pho_s.append(pho_saux)
    a_s.append(a_saux)

df['m_rdlim'] = m_rdlim
df['a_s'] = a_s
df['pho_s'] = pho_s
df

,h,b_w,f_ck,m_rdlim,a_s,pho_s
0,0.3,0.140,20000.0,36.584136,0.000380,0.904886
1,0.3,0.140,35000.0,64.022238,0.000665,1.583550
2,0.3,0.140,50000.0,91.460340,0.000950,2.262214
3,0.3,0.245,20000.0,64.022238,0.000665,0.904886
4,0.3,0.245,35000.0,112.038916,0.001164,1.583550
5,0.3,0.245,50000.0,160.055595,0.001663,2.262214
6,0.3,0.350,20000.0,91.460340,0.000950,0.904886
7,0.3,0.350,35000.0,160.055595,0.001663,1.583550
8,0.3,0.350,50000.0,228.650850,0.002375,2.262214
9,0.5,0.140,20000.0,101.622600,0.000633,0.904886


In [4]:
df_au = {'b_w': [0.14], 'h': [0.3], 'f_ck': [20000], 'm_rdlim': [36.584], 'a_s': [0.000380], 'pho_s': [0.9048]}
df = pd.DataFrame(df_au)
df

,b_w,h,f_ck,m_rdlim,a_s,pho_s
0,0.14,0.3,20000,36.584,0.00038,0.9048


# Subdividindo em grupos por classe de densidade de armadura

In [5]:
b_w = []
h = []
f_ck = []
m_rdlim = []
pho_s = []
a_s = []
divs = 1
for indece, linha in df.iterrows():
    aux = linha['pho_s'] / divs
    pho = 0
    for i in range(divs):
        pho += aux
        pho_s.append(pho)
        b_w.append(linha['b_w'])
        h.append(linha['h'])
        f_ck.append(linha['f_ck'])
        m_rdlim.append(linha['m_rdlim'])
        a_s.append(pho * linha['b_w'] * linha['h'] / 100)

df_aux = {'b_w': b_w, 'h': h, 'f_ck': f_ck, 'm_rdlim': m_rdlim, 'a_s': a_s, 'pho_s': pho_s}
df_aux = pd.DataFrame(df_aux)
df_aux['m_rd'] = df_aux.apply(lambda row: momento_resistente_secao_sem_cor(row['a_s'], row['b_w'], row['h'],row['f_ck']), axis=1)
df_aux

0.13971176470588237


,b_w,h,f_ck,m_rdlim,a_s,pho_s,m_rd
0,0.14,0.3,20000.0,36.584,0.00038,0.9048,40.683619


# Gerando o novo dataset

In [6]:
#df_aux.to_excel('tabela_final_dados.xlsx', index=False)

# Análise de confiabilidade no tempo

In [8]:
beta = []
for i, row in df_aux.iterrows():
    b_w_aux = row['b_w']
    h_aux = row['h']
    f_ck_aux = row['f_ck']
    m_rd_aux = row['m_rd']
    a_s_aux = row['a_s']
    chi_list = [0.3]
    gamma_g = 1.40
    gamma_q = 1.40
    #dados_viga = {'h (m)': h_aux, 'b_w (m)': b_w_aux, 'm_rd (kN.m)': m_rd_aux, 'a_s (m2)': a_s_aux, 'gamma_c': 1.00, 'gamma_s': 1.00, 'gamma_f': 1.00}
    dados_viga = {'d_b (m)': 8/1000, 'd_linha (m)': 3.9/100,'n_b': 3, 'gamma_c': 1.00, 'gamma_s': 1.00, 'gamma_f': 1.00,'b_w (m)': 0.2, 'h (m)': 0.50, 'cob (m)': 0.025, 'ano_construcao': 2000}
    dados_corrosao = {'k_c': 30.5, 'k_fc': 1.7, 'a_d': 0, 'k_ad': 0.32, 'k_co2': 15.5, 'k_rh': 1300, 'k_ce': 1.3}
    none_variable = {'dados_viga': dados_viga, 'time analysis': list(range(0, 101, 10))}

    for id, chi in enumerate(chi_list):
        den_g = gamma_g + gamma_q*chi/(1-chi)
        den_q = gamma_g*(1-chi)/chi + gamma_q
        m_gk = m_rd_aux/den_g
        m_qk = m_rd_aux/den_q

        # Data
        g = {'type': 'normal', 'loc': 1.06*m_gk, 'scale': 0.12*1.06*m_gk, 'stochastic variable': False, 'seed': None}
        q = {'type': 'gumbel max', 'loc': 0.21*m_qk, 'scale': 0.21*0.76*m_qk, 'stochastic variable': True, 'seed': None}
        f_ck = {'type': 'normal', 'loc': 1.22*f_ck_aux, 'scale': 0.15*1.22*f_ck_aux, 'stochastic variable': False, 'seed': None}
        f_yk = {'type': 'normal', 'loc': 1.22*500000, 'scale': 0.04*1.22*500000, 'stochastic variable': False, 'seed': None}
        temp = {'type': 'normal', 'loc': 21.10, 'scale': 0.56, 'stochastic variable': True, 'seed': None}
        u_r = {'type': 'BETA', 'a': 2.938871209812339, 'b': 2.209857158984442, 'loc': 57.250522472205915, 'scale': 14.955723347873345, 'stochastic variable': True, 'seed': None}
        i_corr_20 = {'type': 'lognormal', 'loc': 0.431, 'scale': 0.259, 'stochastic variable': False, 'seed': None}
        teta_r = {'type': 'normal', 'loc': 1, 'scale': 0.05, 'stochastic variable': False, 'seed': None}
        teta_s = {'type': 'normal', 'loc': 1, 'scale': 0.05, 'stochastic variable': False, 'seed': None}
        var = [g, q, f_ck, f_yk, teta_r, teta_s]

        # PAREpy setup
        setup = {
                    'number of samples': 100,
                    'number of dimensions': len(var),
                    'numerical model': {'model sampling': 'mcs-time', 'time steps': len(none_variable['time analysis'])},
                    'variables settings': var,
                    'number of state limit functions or constraints': 1,
                    'none variable': none_variable,
                    'objective function': obj_mestrado_victor,
                    'type process': 'auto',
                    'name simulation': 'victor_hello_world',
                }
        # Call algorithm
        results_aux, pf_aux, beta_aux = sampling_algorithm_structural_analysis(setup)

        # Assembly results
        dic = {}
        dic['chi'] = [chi] * len(none_variable['tempos reais'])
        dic['time'] = none_variable['tempos reais']
        dic['pf'] = pf[0]
        dic['beta'] = beta[0]
        data = pd.DataFrame(dic, columns=['chi', 'time', 'pf', 'beta'])
        df_data.append(data)
        
        # Supondo que você tenha um DataFrame chamado df_aux
        results_aux.to_excel("df_aux.xlsx", index=False)


15:04:26 - Checking inputs completed!
15:04:26 - Started State Limit Function evaluation (g)...
Error: 'a_s (m2)'


KeyError: 'tempos reais'